# 2D convolution accelerator 

Run on the PYNQ board after loading conv2d_design.bit.

Demonstrates:
  1. Loading 3x3 kernel coefficients through AXI-Lite registers
  2. Sending a 5x5 image through AXI DMA
  3. Receiving a 7x7 full-convolution output
  4. Comparing against a Python reference model
  5. Optional AXI Timer measurement

## 1. Load Overlay

In [ ]:
import time
import numpy as np
from pynq import Overlay, allocate

ol = Overlay("conv2d_design.bit")
print("Overlay loaded successfully")
print("IP blocks:")
for name in ol.ip_dict.keys():
    print("  ", name)

dma = ol.axi_dma_0
conv2d = ol.conv2d_stream_0
hw_timer = ol.axi_timer_0


## 2. Problem Dimensions


In [ ]:
IH = 5
IW = 5
KH = 3
KW = 3
OH = IH + KH - 1   # 7
OW = IW + KW - 1   # 7

N_INPUT_REAL = IH * IW       # 25
N_INPUT_STREAM = 7 * 5       # 35, because your HLS currently expects 2 zero rows after image
N_OUTPUT = OH * OW           # 49

## 3. MMIO FIR (one sample per round-trip)

Each sample requires: write input $\rightarrow$ start IP $\rightarrow$ poll done $\rightarrow$ read output.

**Note:** Check the register offsets against the HLS driver header
(`xfir_mmio_hw.h`) after synthesis.

In [ ]:
# Register offsets (from HLS synthesis report -- verify after build)
CTRL_REG       = 0x00
X_IN_OFFSET    = 0x18
RETURN_OFFSET  = 0x10

def float_to_uint(f):
    return struct.unpack('<I', struct.pack('<f', f))[0]

def uint_to_float(u):
    return struct.unpack('<f', struct.pack('<I', u & 0xFFFFFFFF))[0]

def mmio_fir_one_sample(ip, x_val):
    ip.write(X_IN_OFFSET, float_to_uint(x_val))
    ip.write(CTRL_REG, 0x01)                    # ap_start
    while (ip.read(CTRL_REG) & 0x02) == 0:      # wait ap_done
        pass
    return uint_to_float(ip.read(RETURN_OFFSET))

In [ ]:
y_mmio = np.zeros(N_SAMPLES, dtype=np.float32)

t0 = time.perf_counter()
for i in range(N_SAMPLES):
    y_mmio[i] = mmio_fir_one_sample(fir_mmio_ip, float(x[i]))
t_mmio = time.perf_counter() - t0

print(f"MMIO: {N_SAMPLES} samples in {t_mmio*1e3:.2f} ms "
      f"({t_mmio/N_SAMPLES*1e6:.1f} us/sample)")

## 4. DMA Streaming FIR

Data path: PS $\rightarrow$ DMA TX $\rightarrow$ AXI-Stream $\rightarrow$ FIR $\rightarrow$ AXI-Stream $\rightarrow$ DMA RX $\rightarrow$ PS

The CPU only sets up the transfer and waits -- the FPGA processes all samples autonomously.

In [ ]:
in_buf  = allocate(shape=(N_SAMPLES,), dtype=np.float32)
out_buf = allocate(shape=(N_SAMPLES,), dtype=np.float32)

np.copyto(in_buf, x)
out_buf[:] = 0

# Configure streaming FIR: set sample count and start
# Register 0x10 = 'n' parameter (check synthesis report)
fir_stream.write(0x10, N_SAMPLES)
fir_stream.write(0x00, 0x01)   # ap_start

t0 = time.perf_counter()
dma.sendchannel.transfer(in_buf)
dma.recvchannel.transfer(out_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()
t_dma = time.perf_counter() - t0

y_dma = np.array(out_buf, dtype=np.float32)
print(f"DMA: {N_SAMPLES} samples in {t_dma*1e3:.2f} ms "
      f"({t_dma/N_SAMPLES*1e6:.1f} us/sample)")

## 5. AXI Timer (cycle-accurate HW timing)

The AXI Timer counts PL clock cycles. This measures only the
HW processing time (including DMA transfer), without Python overhead.

In [ ]:
TCSR0, TLR0, TCR0 = 0x00, 0x04, 0x08
FCLK_MHZ = 100.0

def timer_start(tmr):
    tmr.write(TLR0, 0)
    tmr.write(TCSR0, 0x020)   # load
    tmr.write(TCSR0, 0x080)   # enable, count up

def timer_stop(tmr):
    cycles = tmr.read(TCR0)
    tmr.write(TCSR0, 0x000)
    return cycles

In [ ]:
# Re-run DMA test with HW timer
np.copyto(in_buf, x)
out_buf[:] = 0

fir_stream.write(0x10, N_SAMPLES)
fir_stream.write(0x00, 0x01)

timer_start(hw_timer)
dma.sendchannel.transfer(in_buf)
dma.recvchannel.transfer(out_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()
cycles = timer_stop(hw_timer)

print(f"HW timer: {cycles} cycles = {cycles/FCLK_MHZ:.1f} us @ {FCLK_MHZ:.0f} MHz")

## 6. Comparison

In [ ]:
print(f"{'Method':<12} {'Time (ms)':>10} {'Speedup':>10}")
print("-" * 34)
print(f"{'MMIO':<12} {t_mmio*1e3:>10.2f} {'1.0x':>10}")
print(f"{'DMA':<12} {t_dma*1e3:>10.2f} {t_mmio/t_dma:>9.1f}x")
print(f"{'HW cycles':<12} {cycles/FCLK_MHZ/1e3:>10.3f} {'(PL only)':>10}")
print()

max_diff = np.max(np.abs(y_dma - y_mmio))
print(f"Max |DMA - MMIO| = {max_diff:.2e}")
assert max_diff < 1e-4, "Output mismatch!"

## 7. Plot Results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(t * 1e3, x, label="Input (50+300 Hz)")
axes[0].set_ylabel("Amplitude")
axes[0].set_title("FIR Filter Input")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t * 1e3, y_dma, label="DMA output")
axes[1].plot(t * 1e3, y_mmio, '--', label="MMIO output", alpha=0.7)
axes[1].set_xlabel("Time (ms)")
axes[1].set_ylabel("Amplitude")
axes[1].set_title("FIR Filter Output (300 Hz suppressed)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Cleanup

In [ ]:
in_buf.freebuffer()
out_buf.freebuffer()